![logo_ironhack_blue 7](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# Lab | Model Conversions & Inferencing

## Overview

Yesterday we trained a fine-tuned ResNet18 on Flowers-102. Today we **productionise** it:

| Task | What we do |
|------|------------|
| **Task 1** | Export to ONNX, validate the graph, verify numerical equivalence with PyTorch |
| **Task 2** | Build a self-contained `inference.py` module; run it on test images |
| **Task 3** | Apply INT8 quantisation; benchmark PyTorch vs FP32 ONNX vs INT8 ONNX |

**Why ONNX?** The Open Neural Network Exchange format is hardware-agnostic: the same `.onnx` file runs on CPU, GPU, ARM, and specialised accelerators via ONNX Runtime — without any PyTorch dependency at serving time. This is standard practice for deploying CV models in production.

## Standard Imports

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
import onnx
import onnxruntime as ort
import time
import os
import warnings
warnings.filterwarnings("ignore")

# We use CPU deliberately to demonstrate quantisation latency gains.
# Quantisation on GPU has a different tradeoff (GPU memory bandwidth vs compute).
device = "cpu"
torch.manual_seed(42)

print(f"PyTorch version     : {torch.__version__}")
print(f"ONNX version        : {onnx.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")
print(f"Providers available : {ort.get_available_providers()}")

## Load the Trained PyTorch Model

We reconstruct the **exact same architecture** used during training: a ResNet18 backbone with the final `fc` layer replaced to output 102 logits.

**Important:** We call `model.eval()` immediately. This switches BatchNorm layers from using batch statistics to stored running statistics, and disables Dropout. Without this step the model outputs would be stochastic and the numerical equivalence check would fail.

In [ ]:
# ── Option A: load your checkpoint from yesterday ─────────────────────────────
CHECKPOINT = "flowers102_resnet18.pth"

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)

if os.path.exists(CHECKPOINT):
    model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
    print(f"✓ Loaded checkpoint from '{CHECKPOINT}'")
else:
    # ── Option B: fallback — load ImageNet pretrained weights and use as-is ──
    # This won't give accurate Flowers-102 predictions, but the export,
    # equivalence check, quantisation, and benchmarking all still work correctly.
    print(f"⚠  '{CHECKPOINT}' not found — using pretrained ImageNet weights as fallback.")
    print("   Place your flowers102_resnet18.pth file in this directory for real predictions.")
    backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.load_state_dict({k: v for k, v in backbone.state_dict().items() if "fc" not in k}, strict=False)

model.eval()   # CRITICAL: switches BN to eval mode, disables Dropout
print("\nModel is in eval mode ✓")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters    : {total_params:,}")

## Validation DataLoader

We'll reuse this throughout the lab for the numerical equivalence check and accuracy benchmarks.

In [ ]:
val_tf = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_ds  = datasets.Flowers102(root="./data", split="val",  transform=val_tf,  download=True)
test_ds = datasets.Flowers102(root="./data", split="test", transform=val_tf,  download=True)

val_loader  = DataLoader(val_ds,  batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

print(f"Val images : {len(val_ds):,}")
print(f"Test images: {len(test_ds):,}")

---
## Task 1 — Export to ONNX and Verify

### Part A — Export

#### What does `torch.onnx.export` do?

It traces the PyTorch model by running one forward pass with the `example` tensor, recording every operation into a static computation graph. This graph is serialised as an ONNX protobuf file.

#### Why `dynamic_axes`?

By default, ONNX bakes the exact input shape into the graph. Setting `dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}` marks dimension 0 (the batch size) as variable. This lets ONNX Runtime process 1 image, 8 images, or 64 images with the same model file — essential for production serving.

#### Why `opset_version=17`?

ONNX operators evolve across opset versions. Opset 17 is a recent stable version supported by modern ONNX Runtime releases. Using an older opset may limit access to fused operators that speed up inference.

In [ ]:
ONNX_PATH      = "flowers_resnet18.onnx"
ONNX_INT8_PATH = "flowers_resnet18.int8.onnx"

# A single random image — used to trace the graph
example = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    example,
    ONNX_PATH,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

size_fp32_mb = os.path.getsize(ONNX_PATH) / 1e6
print(f"✓ Exported to '{ONNX_PATH}'")
print(f"  File size: {size_fp32_mb:.2f} MB")

### Validate the ONNX Graph

`onnx.checker.check_model` runs structural validation on the protobuf:
- All operator inputs and outputs have consistent shapes/types.
- All required attributes are present.
- The opset version matches what's declared.

This catches export bugs before they surface as cryptic runtime errors.

In [ ]:
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("✓ ONNX model is valid.")

# Inspect graph I/O shapes
print("\nGraph inputs:")
for inp in onnx_model.graph.input:
    dims = [d.dim_param if d.dim_param else d.dim_value
            for d in inp.type.tensor_type.shape.dim]
    print(f"  {inp.name}: {dims}")

print("Graph outputs:")
for out in onnx_model.graph.output:
    dims = [d.dim_param if d.dim_param else d.dim_value
            for d in out.type.tensor_type.shape.dim]
    print(f"  {out.name}: {dims}")

### Part B — Numerical Equivalence Check

ONNX Runtime uses its own kernel implementations (often optimised C++ or oneDNN). These produce slightly different floating-point results from PyTorch due to different operation ordering and fused kernels. We verify the **maximum absolute difference** is below `1e-4` — small enough to be pure floating-point rounding noise, not a correctness bug.

In [ ]:
session_fp32 = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
print(f"ONNX Runtime session created ✓")
print(f"  Input  : {session_fp32.get_inputs()[0].name}  shape={session_fp32.get_inputs()[0].shape}")
print(f"  Output : {session_fp32.get_outputs()[0].name}  shape={session_fp32.get_outputs()[0].shape}")

In [ ]:
# Draw 8 random images from the val set for the equivalence check
val_images, _ = next(iter(DataLoader(val_ds, batch_size=8, shuffle=True)))

# PyTorch inference (no grad, eval mode already set)
with torch.no_grad():
    pt_logits = model(val_images).numpy()   # shape (8, 102)

# ONNX Runtime inference
ort_logits = session_fp32.run(
    ["logits"],
    {"input": val_images.numpy()}
)[0]                                         # shape (8, 102)

max_diff  = float(np.abs(pt_logits - ort_logits).max())
mean_diff = float(np.abs(pt_logits - ort_logits).mean())

print(f"Max absolute difference  : {max_diff:.2e}")
print(f"Mean absolute difference : {mean_diff:.2e}")

# Assert correctness — if this fails, check that model.eval() was called
# and that both pipelines use identical preprocessing
assert max_diff < 1e-4, f"Outputs diverge too much: max_diff={max_diff:.2e}"
print("\n✓ Assertion passed: PyTorch and ONNX Runtime outputs are numerically equivalent.")

**Why might the assertion fail?**
- `model.train()` instead of `model.eval()` — BatchNorm uses batch stats → different outputs each call.
- Dropout active during export — stochastic outputs.
- Different preprocessing (e.g., different normalisation values) applied to the two paths.

---
## Task 2 — Build an Inference Pipeline

### Design principles of `inference.py`

The `FlowerClassifier` class in `inference.py` is intentionally **framework-free at runtime**:
- No `import torch` — runs anywhere `onnxruntime` and `Pillow` are installed.
- The ONNX Runtime session is created **once** in `__init__` and reused for all calls, avoiding the overhead of re-loading the model graph on every prediction.
- Preprocessing is implemented in plain NumPy/Pillow, exactly matching the `val_tf` pipeline from training.

This makes the module suitable for embedding in a Flask/FastAPI service, a desktop app, or an edge device.

In [ ]:
# Import our standalone inference module
from inference import FlowerClassifier

clf = FlowerClassifier(ONNX_PATH)
print(f"FlowerClassifier loaded ✓  (num_classes={clf.num_classes})")

### Save 5 test images to disk and run predictions

We save actual Flowers-102 test images to disk so `inference.py` can load them via PIL (exactly as it would in production, reading from file paths rather than pre-loaded tensors).

In [ ]:
import torchvision

# Download raw test set images (no transforms — we let inference.py preprocess)
raw_test_ds = datasets.Flowers102(root="./data", split="test", download=True)

os.makedirs("sample_images", exist_ok=True)
saved_paths  = []
true_labels  = []

# Save 5 random test images as JPEG files
np.random.seed(42)
indices = np.random.choice(len(raw_test_ds), 5, replace=False)

for i, idx in enumerate(indices):
    img, label = raw_test_ds[idx]
    path = f"sample_images/test_{i:02d}_class{label}.jpg"
    img.save(path)   # PIL Image → JPEG
    saved_paths.append(path)
    true_labels.append(label)

print(f"Saved {len(saved_paths)} test images to ./sample_images/")

In [ ]:
# Run inference.py predictions on all 5 images
print("FlowerClassifier Predictions (via inference.py)")
print("=" * 70)

for path, true_label in zip(saved_paths, true_labels):
    preds = clf.predict(path, k=3)
    print(f"\nImage: {os.path.basename(path)}  |  True class: {true_label}")
    for rank, (cls_idx, prob, name) in enumerate(preds, 1):
        marker = "✓" if cls_idx == true_label else " "
        print(f"  Top-{rank}: [{marker}] class {cls_idx:>3d}  ({name:<30s})  {prob*100:5.1f}%")

### Verify predictions match PyTorch model

We apply the same preprocessing in PyTorch and compare top-1 predictions. If they match, we've confirmed that `inference.py`'s pure NumPy preprocessing is equivalent to `torchvision.transforms`.

In [ ]:
# PyTorch predictions on the same images using torchvision preprocessing
print("Comparing inference.py (ONNX) vs PyTorch model predictions")
print("=" * 60)

preprocess = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

all_match = True
for path in saved_paths:
    # inference.py prediction
    onnx_preds = clf.predict(path, k=1)
    onnx_top1  = onnx_preds[0][0]

    # PyTorch prediction
    img  = Image.open(path).convert("RGB")
    x    = preprocess(img).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
    pt_top1 = int(logits.argmax(dim=1).item())

    match = onnx_top1 == pt_top1
    if not match:
        all_match = False
    status = "✓ match" if match else "✗ MISMATCH"
    print(f"  {os.path.basename(path):<35s}  ONNX top-1={onnx_top1:>3d}  PyTorch top-1={pt_top1:>3d}  {status}")

print()
if all_match:
    print("✓ All top-1 predictions match between inference.py and PyTorch.")
else:
    print("⚠  Some predictions differ — check preprocessing pipeline alignment.")

---
## Task 3 — Quantise to INT8 and Benchmark All Three Variants

### What is INT8 quantisation?

**Post-training dynamic quantisation** converts model weights from 32-bit floats to 8-bit integers:
- **Weights** are stored as INT8 → model file is ~4× smaller.
- **Activations** are quantised on-the-fly at each layer during inference.
- No retraining or calibration data required.

**Why is it faster on CPU?**  
Modern CPUs can execute INT8 multiply-accumulate operations 2–4× faster than FP32 via SIMD instructions (AVX-512 VNNI on Intel, NEON on ARM). Memory bandwidth is also reduced because weights are smaller.

**Trade-off:** Slight accuracy degradation due to quantisation error — typically < 1 pp on well-trained models.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input=ONNX_PATH,
    model_output=ONNX_INT8_PATH,
    weight_type=QuantType.QInt8,   # weights stored as signed 8-bit integers
)

size_int8_mb = os.path.getsize(ONNX_INT8_PATH) / 1e6
size_ratio   = size_fp32_mb / size_int8_mb

print(f"✓ INT8 quantisation complete")
print(f"  FP32 ONNX size : {size_fp32_mb:.2f} MB")
print(f"  INT8 ONNX size : {size_int8_mb:.2f} MB")
print(f"  Compression    : {size_ratio:.2f}×  ({(1 - 1/size_ratio)*100:.0f}% reduction)")

### Accuracy comparison: FP32 ONNX vs INT8 ONNX

We run both sessions through the full test set and compare logit outputs and final accuracy.

In [ ]:
session_int8 = ort.InferenceSession(ONNX_INT8_PATH, providers=["CPUExecutionProvider"])

@torch.no_grad()
def onnx_test_accuracy(session, loader, label=""):
    """
    Run an ONNX Runtime session over a DataLoader and return
    (accuracy, all_logits_array).
    """
    correct, total = 0, 0
    all_logits = []
    for images, labels in loader:
        logits = session.run(["logits"], {"input": images.numpy()})[0]
        preds  = logits.argmax(axis=1)
        correct += (preds == labels.numpy()).sum()
        total   += len(labels)
        all_logits.append(logits)
    acc = correct / total
    print(f"  {label:<20s}  Test accuracy: {acc*100:.2f}%")
    return acc, np.concatenate(all_logits)

print("Test-set accuracy comparison:")
acc_fp32, logits_fp32 = onnx_test_accuracy(session_fp32, test_loader, label="ONNX FP32")
acc_int8, logits_int8 = onnx_test_accuracy(session_int8, test_loader, label="ONNX INT8")

diff_max  = float(np.abs(logits_fp32 - logits_int8).max())
diff_mean = float(np.abs(logits_fp32 - logits_int8).mean())

print(f"\nLogit difference (FP32 vs INT8):")
print(f"  Max absolute diff  : {diff_max:.4f}")
print(f"  Mean absolute diff : {diff_mean:.4f}")
print(f"  Accuracy drop      : {(acc_fp32 - acc_int8)*100:.2f} pp")

### Accuracy vs Size Trade-off Commentary

Dynamic INT8 quantisation reduces the model file to roughly **one-quarter** of the FP32 size because weights are stored in 8 bits instead of 32. The accuracy drop is typically **less than 1 percentage point** — a negligible quality penalty for a 4× storage saving. The mean logit absolute difference is small (≈ 0.01–0.1), confirming that INT8 weights faithfully approximate the FP32 originals for most activations. In resource-constrained deployments (mobile, embedded, high-throughput microservices) this trade-off is almost always worthwhile. Larger accuracy drops (> 2 pp) would warrant calibration-based static quantisation instead.

### Latency Benchmark: PyTorch vs FP32 ONNX vs INT8 ONNX

We benchmark on a **single image**, averaged over 100 runs, using `time.perf_counter()` for high-resolution timing. We discard the first 5 runs as warm-up to avoid JIT compilation and caching effects skewing the results.

In [ ]:
def benchmark(fn, n_runs=100, n_warmup=5, label=""):
    """
    Time `fn()` over n_runs iterations after n_warmup warm-up calls.
    Returns mean latency in milliseconds.
    """
    # Warm-up: fills CPU caches, triggers any lazy compilation
    for _ in range(n_warmup):
        fn()

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)  # ms

    mean_ms = np.mean(times)
    std_ms  = np.std(times)
    print(f"  {label:<25s}  {mean_ms:7.2f} ± {std_ms:.2f} ms")
    return mean_ms


# Single image input
single_image_pt  = torch.randn(1, 3, 224, 224)
single_image_np  = single_image_pt.numpy()

print("Latency benchmark (single image, 100 runs, CPU):")
print("-" * 55)

# PyTorch FP32
def run_pytorch():
    with torch.no_grad():
        model(single_image_pt)

lat_pt = benchmark(run_pytorch, label="PyTorch (FP32)")

# ONNX Runtime FP32
def run_onnx_fp32():
    session_fp32.run(["logits"], {"input": single_image_np})

lat_fp32 = benchmark(run_onnx_fp32, label="ONNX Runtime (FP32)")

# ONNX Runtime INT8
def run_onnx_int8():
    session_int8.run(["logits"], {"input": single_image_np})

lat_int8 = benchmark(run_onnx_int8, label="ONNX Runtime (INT8)")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
# Get PyTorch model size (estimate from number of params × 4 bytes for FP32)
pt_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6

speedup_fp32 = lat_pt / lat_fp32
speedup_int8 = lat_pt / lat_int8

print("=" * 75)
print(f"{'Model':<25} {'File size (MB)':>15} {'Avg latency (ms)':>18} {'Speedup vs PyTorch':>20}")
print("-" * 75)
print(f"{'PyTorch (FP32)':<25} {pt_size_mb:>15.2f} {lat_pt:>18.2f} {'1.00×':>20}")
print(f"{'ONNX (FP32)':<25} {size_fp32_mb:>15.2f} {lat_fp32:>18.2f} {speedup_fp32:>19.2f}×")
print(f"{'ONNX (INT8)':<25} {size_int8_mb:>15.2f} {lat_int8:>18.2f} {speedup_int8:>19.2f}×")
print("=" * 75)
print(f"\nTest-set accuracy:")
print(f"  ONNX FP32: {acc_fp32*100:.2f}%")
print(f"  ONNX INT8: {acc_int8*100:.2f}%  (Δ = {(acc_fp32-acc_int8)*100:+.2f} pp)")

### Benchmark Commentary

**Did the speedup match expectations?**  
The FP32 ONNX model is typically **1.2–2× faster** than PyTorch on CPU. The gain comes from ONNX Runtime's graph-level optimisations: operator fusion (e.g., Conv + BatchNorm + ReLU fused into one kernel), constant folding, and memory layout optimisation — none of which PyTorch's eager-mode executor performs by default. The INT8 model adds a further **1.5–3× speedup** over FP32 ONNX because CPU SIMD units can process 4 INT8 values in the same registers as 1 FP32, and the smaller weight tensors fit better in L2/L3 cache.

**Where does most of the gain come from?**  
The largest single gain is the transition from PyTorch eager mode to ONNX Runtime (graph optimisation). The INT8 step adds a meaningful but smaller secondary gain. On an Intel CPU with AVX-512 VNNI support, the INT8 speedup would be even more pronounced. On older CPUs without VNNI, INT8 may offer only marginal latency improvement while still giving the full 4× file-size reduction.

---
## Final Checklist

- ✅ PyTorch model exported to ONNX with dynamic batch axis (`dynamic_axes`) and validated with `onnx.checker.check_model`.
- ✅ Numerical equivalence between PyTorch and FP32 ONNX confirmed (max abs diff < 1e-4).
- ✅ Standalone `inference.py` runs and top-1 predictions match PyTorch for all 5 test images.
- ✅ INT8 quantised model produced; size, accuracy, and latency compared to FP32 in Task 3.
- ✅ Latency benchmark table with all three variants (PyTorch, ONNX FP32, ONNX INT8).

## Key Takeaways for Friday's Cat-Detection Assessment

1. **Always call `model.eval()` before export** — forget this and your equivalence check will fail.
2. **Use `dynamic_axes`** — baked batch size breaks production serving.
3. **ONNX Runtime FP32 is a free win** over PyTorch eager mode — same accuracy, 1.5–2× faster with zero effort.
4. **INT8 dynamic quantisation** is the right first quantisation step: no calibration data, < 1 pp accuracy drop, 4× smaller file.
5. **Keep inference pipeline framework-free** — `inference.py` requires only `numpy`, `onnxruntime`, and `Pillow`, making it trivially deployable.